In [1]:
import pandas as pd


In [4]:
df = pd.read_csv("../dataset/base_raw.csv")
print(df.columns)

Index(['age', 'insurance_type', 'admission_type', 'admission_source',
       'length_of_stay', 'department', 'room_type', 'num_procedures',
       'num_medications', 'has_diabetes', 'has_hypertension',
       'has_heart_disease', 'has_copd', 'has_kidney_disease',
       'previous_admissions', 'discharge_systolic_bp',
       'discharge_diastolic_bp', 'discharge_heart_rate',
       'discharge_temperature', 'discharge_o2_saturation', 'hemoglobin_level',
       'wbc_count', 'creatinine_level', 'glucose_level',
       'discharge_disposition', 'follow_up_scheduled',
       'medication_adherence_score', 'readmitted_30days'],
      dtype='object')


In [ ]:
# H₀:Diabetes has no effect on readmission


import pandas as pd
from scipy.stats import chi2_contingency

# -------------------------------
# 1. Define categorical columns
# -------------------------------
cat_cols = [
    # 'insurance_type','admission_type','admission_source',
    # 'department','room_type',
    'has_diabetes'
    ,
    # 'has_hypertension','has_heart_disease',
    # 'has_copd','has_kidney_disease',
    # 'discharge_disposition','follow_up_scheduled'
]

target = 'readmitted_30days'

# -------------------------------
# 2. Store results
# -------------------------------
results = []

# -------------------------------
# 3. Loop through each feature
# -------------------------------
for col in cat_cols:
    
    print("\n" + "="*50)
    print(f"Column: {col}")
    print("="*50)
    
    # Create contingency table
    table = pd.crosstab(df[col], df[target])
    
    print("\n--- Contingency Table ---")
    print(table)
    
    # Apply Chi-square test
    chi2, p, dof, expected = chi2_contingency(table)
    
    print("\nChi2 value:", chi2)
    print("P-value:", p)
    
    # Decision
    if p < 0.05:
        print("Result: ✅ Significant (Reject H0)")
        significance = "Significant"
    else:
        print("Result: ❌ Not Significant (Fail to Reject H0)")
        significance = "Not Significant"
    
    # -------------------------------
    # 4. Readmission Rate Analysis
    # -------------------------------
    print("\n--- Readmission Rates ---")
    
    rate = table.div(table.sum(axis=1), axis=0)
    print(rate)
    
    # Save results
    results.append({
        "feature": col,
        "p_value": p,
        "chi2": chi2,
        "result": significance
    })

# -------------------------------
# 5. Final Summary Table
# -------------------------------
results_df = pd.DataFrame(results)

print("\n\n========== FINAL SUMMARY ==========")
print(results_df.sort_values(by="p_value"))

# -------------------------------
# 6. Save results
# -------------------------------
results_df.to_csv("chi_square_results.csv", index=False)


Column: has_diabetes

--- Contingency Table ---
readmitted_30days    No   Yes
has_diabetes                 
No                 8011  2279
Yes                2602  1388

Chi2 value: 240.00162069464088
P-value: 3.929634349596226e-54
Result: ✅ Significant (Reject H0)

--- Readmission Rates ---
readmitted_30days        No       Yes
has_diabetes                         
No                 0.778523  0.221477
Yes                0.652130  0.347870


========== FINAL SUMMARY ==========
        feature       p_value        chi2       result
0  has_diabetes  3.929634e-54  240.001621  Significant
